In [ ]:
from pathlib import Path

import polars as pl
import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

from climate_attitudes.extract.dataset import ClimateAttitudesDataset

ASSETS_DIR = Path("/Users/henry/data/msc_thesis/climate-attitudes")

In [ ]:
data = ClimateAttitudesDataset(ASSETS_DIR)

In [ ]:
TOPICS = [
    "lee_2025_cc_happening",
    "lee_2025_cc_human",
    "lee_2025_cc_worried",
    "lee_2025_personal_harm",
    "lee_2025_future_gen_harm",
    "lee_2025_fossil_fuel_reduction",
    "lee_2025_renewable_energy",
    "lee_2025_govt_priority",
]

# Human-readable
TOPICS_HR = [
    "Belief: CC happening",
    "Belief: Anthropogenic CC",
    "Attitude: CC worry",
    "Expectation: CC personal harm",
    "Expectation: CC future gen. harm",
    "Issue position: Fossil fuel reduction",
    "Issue position: Renewable energy investment",
    "Issue position: CC as govt. priority",
]

In [ ]:
data.item.filter(pl.col("name").str.starts_with("cc_pol"))

In [ ]:
data.item

In [ ]:
relevant_items = data.item.select(pl.col("name"), pl.col(r"^lee_.*$")).filter(
    pl.any_horizontal(pl.col(r"^lee_.*$"))
)
with pl.Config(tbl_rows=121):
    print(relevant_items.select(pl.col("name")).to_series())

In [ ]:
for topic in TOPICS:
    items = data.item.filter(
        pl.col(topic),
    ).select(pl.col("name").unique(maintain_order=True))
    print(items)

Heatmap showing item association for each topic (irrespective of wave)

In [ ]:
relevant_items = (
    data.item.select(pl.col("name"), pl.col(r"^lee_.*$"))
    .filter(pl.any_horizontal(pl.col(r"^lee_.*$")))
    .filter(
        ~(pl.col("name") == "cvcc6"),
        ~pl.col("name").str.starts_with("ccComp"),
        ~pl.col("name").str.starts_with("pol_vote_CC"),
    )
)

assoc = relevant_items.drop("name").to_numpy().T

# corr = question_response.filter(pl.col("wave") == 1).drop("participant_id", "response_id", "wave").to_pandas().corr()

# Generate a mask for the upper triangle
# mask = np.triu(np.ones_like(corr, dtype=bool))

# Set up the matplotlib figure
f, axe = plt.subplots(figsize=(20, 3))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(
    assoc,
    cmap=cmap,
    center=0,
    square=True,
    linewidths=0.5,
    cbar=False,
    cbar_kws={"shrink": 0.5},
)

axe.set_yticklabels(TOPICS_HR)
axe.set_xticks(
    np.arrange(len(relevant_items)) + 0.5, relevant_items.select("name").to_series()
)
axe.tick_params(axis="y", labelrotation=0)
axe.tick_params(axis="x", labelrotation=90)

For each topic, show a heatmap of the question occurrence by wave

In [ ]:
for topic in TOPICS:
    relevant_items = (
        data.item.select(pl.col("item_id"), pl.col("name"), pl.col(topic))
        .filter(pl.col(topic))
        .filter(
            ~(pl.col("name") == "cvcc6"),
            ~pl.col("name").str.starts_with("ccComp"),
            ~pl.col("name").str.starts_with("pol_vote_CC"),
            ~(
                pl.col("name") == "cc13"
            ),  # Error in codebook, two different questions for wave 1
            ~(
                pl.col("name") == "cc_commit"
            ),  # Different questions in wave 2 for new/repeating participants
        )
        .join(data.question, on="item_id", how="left")
        .select("name", "wave", topic)
        .pivot("wave", values=topic)
    )
    for colname in ("1", "2", "3", "4", "5"):
        if colname not in relevant_items.columns:
            relevant_items = relevant_items.with_columns(pl.lit(False).alias(colname))
    relevant_items = relevant_items.with_columns(
        pl.col("1", "2", "3", "4", "5").fill_null(False)
    )

    assoc = relevant_items.drop("name").to_numpy().T

    # Set up the matplotlib figure
    f, axe = plt.subplots(figsize=(20, 3))

    # Generate a custom diverging colormap
    cmap = sns.diverging_palette(230, 20, as_cmap=True)

    # Draw the heatmap with the mask and correct aspect ratio
    sns.heatmap(
        assoc,
        cmap=cmap,
        center=0,
        square=True,
        linewidths=0.5,
        cbar=False,
        cbar_kws={"shrink": 0.5},
    )

    axe.set_xticks(
        np.arrange(len(relevant_items)) + 0.5, relevant_items.select("name").to_series()
    )
    axe.tick_params(axis="y", labelrotation=0)
    axe.tick_params(axis="x", labelrotation=90)

    axe.set_title(f"Relevant survey items by wave: {topic}")

    plt.show()

In [ ]:
pl.read_parquet(
    "/Users/henry/data/msc_thesis/climate-attitudes-constructed/lee_2025_items.parquet"
)

For each wave, show the heatmap of question presence per topic

In [ ]:
relevant_items = (
    data.item.select(pl.col("name"), pl.col(r"^lee_.*$"))
    .filter(pl.any_horizontal(pl.col(r"^lee_.*$")))
    .filter(
        ~(pl.col("name") == "cvcc6"),
        ~pl.col("name").str.starts_with("ccComp"),
        ~pl.col("name").str.starts_with("pol_vote_CC"),
    )
)

wave = 1

In [ ]:
all_relevant_items = (
    data.item.select(pl.col("item_id"), pl.col("name"), pl.col(r"^lee_.*$"))
    .filter(pl.any_horizontal(pl.col(r"^lee_.*$")))
    .filter(
        ~(pl.col("name") == "cvcc6"),
        ~pl.col("name").str.starts_with("ccComp"),
        ~pl.col("name").str.starts_with("pol_vote_CC"),
        ~(
            pl.col("name") == "cc13"
        ),  # Error in codebook, two different questions for wave 1
        ~(
            pl.col("name") == "cc_commit"
        ),  # Different questions in wave 2 for new/repeating participants
    )
    .select("item_id", "name")
)

for wave in range(1, 6):
    relevant_items = (
        data.item.select(pl.col("item_id"), pl.col("name"), pl.col(r"^lee_.*$"))
        .filter(pl.any_horizontal(pl.col(r"^lee_.*$")))
        .filter(
            ~(pl.col("name") == "cvcc6"),
            ~pl.col("name").str.starts_with("ccComp"),
            ~pl.col("name").str.starts_with("pol_vote_CC"),
            ~(
                pl.col("name") == "cc13"
            ),  # Error in codebook, two different questions for wave 1
            ~(
                pl.col("name") == "cc_commit"
            ),  # Different questions in wave 2 for new/repeating participants
        )
        .join(data.question, on="item_id", how="left")
        .filter(pl.col("wave") == wave)
        .join(all_relevant_items, on=("item_id", "name"), how="right")
        .with_columns(pl.col(r"^lee_.*$").fill_null(False))
    )

    # for colname in ("1", "2", "3", "4", "5"):
    #     if colname not in relevant_items.columns:
    #         relevant_items = relevant_items.with_columns(pl.lit(False).alias(colname))
    # relevant_items = relevant_items.with_columns(pl.col("1", "2", "3", "4", "5").fill_null(False))

    assoc = relevant_items.select(pl.col(r"^lee_.*$")).to_numpy().T

    # Set up the matplotlib figure
    f, axe = plt.subplots(figsize=(20, 3))

    # Generate a custom diverging colormap
    cmap = sns.diverging_palette(230, 20, as_cmap=True)

    # Draw the heatmap with the mask and correct aspect ratio
    sns.heatmap(
        assoc,
        cmap=cmap,
        center=0,
        square=True,
        linewidths=0.5,
        cbar=False,
        cbar_kws={"shrink": 0.5},
    )

    axe.set_yticklabels(TOPICS_HR)
    axe.set_xticks(
        np.arrange(len(relevant_items)) + 0.5, relevant_items.select("name").to_series()
    )
    axe.tick_params(axis="y", labelrotation=0)
    axe.tick_params(axis="x", labelrotation=90)

    axe.set_title(f"Relevant survey items by topic, wave {wave}")

    plt.show()

For each combination of waves, identify which items are such that they are asked of new participants on the first wave, and repeating participants on all other waves.

In [ ]:
import itertools

In [ ]:
wave_combos = pl.DataFrame(
    {
        "wave_combo": [
            combination
            for i in range(1, 6)
            for combination in itertools.combinations(range(1, 6), i)
        ]
    }
)

# filter to only the relevant questions
relevant_questions = all_relevant_items.select("item_id").join(
    data.question, on="item_id", how="left"
)

# for each item, determine combinations of waves such that it is asked of new participants in first wave, then repeating in all others
item_valid_wave_combos = (
    relevant_questions.join(wave_combos, how="cross")
    .filter(pl.col("wave").is_in(pl.col("wave_combo")))
    .filter(
        (
            (pl.col("wave") == pl.col("wave_combo").list.min())
            & pl.col("new_participants")
        )
        | (
            (pl.col("wave") != pl.col("wave_combo").list.min())
            & pl.col("repeating_participants")
        )
    )
    .group_by("item_name", "wave_combo", maintain_order=True)
    .agg(pl.col("wave").alias("waves_present"))
    .filter(pl.col("waves_present") == pl.col("wave_combo"))
    .select(pl.col("item_name").alias("name"), "wave_combo")
)

# determine the number of participants who match each wave combination
wave_combo_participant_count = (
    data.participant.select("participant_id", "wave_joined")
    .join(data.response, on="participant_id", how="left")
    .group_by("participant_id")
    .agg("wave")
    .join(item_valid_wave_combos.select(pl.col("wave_combo").unique()), how="cross")
    .filter(
        pl.col("wave_combo").list.set_intersection(pl.col("wave"))
        == pl.col("wave_combo"),  # Participant responded to all waves
        pl.col("wave").list.min()
        == pl.col(
            "wave_combo"
        ).list.min(),  # Participant joined in the first wave of the combo
    )
    .group_by("wave_combo")
    .agg(pl.len().alias("participant_count"))
)

# then join with the item table, to see how the items vary across Lee et al. topics, for valid wave combos
(
    item_valid_wave_combos.join(
        data.item.select(pl.col("item_id"), pl.col("name"), pl.col(r"^lee_.*$")),
        on="name",
        how="left",
    )
    .join(wave_combo_participant_count, on="wave_combo", how="left")
    .filter(pl.col("wave_combo").list.len() > 1)
)